# Signature verification - evaluate Stage 3 base vs augmented checkpoint

Evaluates one or more trained Stage 3 ConvNeXt-Tiny checkpoints on the **grayscale
processed test split** and reports **FAR / FRR at the best-EER operating point** (no
`--target-far`, no score fusion).

The notebook clones the repo, points at the checkpoints you mount, and runs the same
verification protocol as `training.evaluate` (a query's score is its mean cosine
similarity to the signer's enrolled references). It then reads FAR and FRR at the
threshold that makes them equal -- the split's own equal-error point.

**Inputs you attach**

| Input | Where | What |
| --- | --- | --- |
| preprocessed dataset | Kaggle dataset, mounted read-only | `manifest.csv` + image folders, built with `preprocessing.preprocess_dataset --grayscale`, zipped **flat** |
| checkpoints | Kaggle dataset or notebook output | `stage3_base/best_model.pt`, `stage3_aug/best_model.pt` |

Cells run top to bottom: clone -> configure paths -> verify -> evaluate -> compare.

In [ ]:
# Cell 1 - clone the repo and install only what Kaggle is missing.
# Kaggle images already ship torch/torchvision; reinstalling them wastes ~2 GB and time.
# Change REPO_URL if you are evaluating from a different fork.
REPO_URL = "https://github.com/acharyayush/Signature-Verification-FIRST-.git"
!git clone $REPO_URL /kaggle/working/repo
%cd /kaggle/working/repo
# The handcrafted-feature branch is not used here, so albumentations is not needed.
!pip install -q --no-input timm opencv-python scikit-learn

In [ ]:
# Cell 2 - import the repo and check the accelerator.
import os, sys, torch

REPO_DIR = "/kaggle/working/repo"
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)  # so training / scoring / preprocessing import from the clone

print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("gpu  ", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")
print(sys.version)

## 1. Configure the inputs

Edit the two blocks below. `PROCESSED` is the attached preprocessed dataset and `MODELS`
maps a label to each checkpoint you want to evaluate. Add or remove entries freely -- the
comparison table is built from whatever is in `MODELS`.

In [ ]:
# Cell 3 - point at the dataset and the checkpoints.
from pathlib import Path

# Attached preprocessed dataset. manifest.csv sits at the dataset root and image paths
# are stored relative to it, so --image-root defaults to the manifest's directory.
PROCESSED = "/kaggle/input/signature-verification-processed"

# The checkpoints to compare. Point each entry at wherever the .pt file is mounted: a
# Kaggle dataset, or /kaggle/working/repo/checkpoints/... if you just trained in this
# session. stage3_base is the no-augmentation model, stage3_aug the augmented one.
MODELS = {
    "stage3_base": "/kaggle/input/signature-models/stage3_base/best_model.pt",
    "stage3_aug":  "/kaggle/input/signature-models/stage3_aug/best_model.pt",
}

MANIFEST = str(Path(PROCESSED) / "manifest.csv")
REFERENCES = 5    # genuine signatures enrolled per signer before scoring queries
DEVICE = "cuda"   # use "cpu" if no accelerator is attached
BATCH_SIZE = 64

In [ ]:
# Cell 4 - verify the dataset and the checkpoints before the run.
import csv, collections
from pathlib import Path

root = Path(PROCESSED)
rows = list(csv.DictReader(open(MANIFEST)))
print("rows    ", len(rows))
print("splits  ", collections.Counter(r["split"] for r in rows))
print("signers ", len({r["signer_id"] for r in rows}))
print("datasets", collections.Counter(r["image_path"].split("/")[0] for r in rows))

missing = [r["image_path"] for r in rows if not (root / r["image_path"]).exists()]
print("missing images", len(missing), missing[:3])
assert not missing, "dataset incomplete or manifest paths do not match the layout"

for name, path in MODELS.items():
    ok = Path(path).is_file()
    print(f"{name:12s} -> {'OK' if ok else 'MISSING'}  {path}")
    assert ok, f"checkpoint not found: {path}  (edit MODELS in Cell 3)"

## 2. Evaluate at the best-EER operating point

For every model the notebook embeds the **test** split once, scores each query against its
signer's enrolled references, and reports:

* `eer_skilled` / `eer_random` - the equal-error rates for skilled forgeries and for random
  (other-signer) impostors;
* `far_skilled`, `frr` - false-accept and false-reject rates read **at the equal-error
  threshold**, i.e. the FAR / FRR that correspond to the best EER;
* `far_random` - random-impostor FAR at the same threshold.

Random impostors are restricted to **other signers of the same dataset**, the harder and
more meaningful number. No `--target-far` is used and nothing is tuned on the test split:
the EER point is reported as a characterisation of the score separation.

The per-dataset block reuses the same embeddings, so it costs no extra forward passes.

In [ ]:
# Cell 5 - deep-only evaluation, FAR / FRR at the best-EER operating point.
import json
from pathlib import Path

import torch

from scoring.metrics import eer_threshold, rates_at_threshold
from training.evaluate import _evaluate_split
from training.evaluate_per_dataset import _collect
from training.stage3_metric_learning import (
    SiameseSignatureModel,
    dataset_labels,
    embed_records,
    read_split,
)

records = read_split(MANIFEST, PROCESSED, {"test"})
dataset_keys = dataset_labels(records, PROCESSED, 1)
dataset_of = {r.signer_id: key for r, key in zip(records, dataset_keys)}
print(f"{len(records)} test records over {sorted(set(dataset_keys))}")


def rate_block(genuine, skilled, random):
    """FAR / FRR read at the split's own equal-error (best-EER) threshold."""
    threshold, eer_skilled = eer_threshold(genuine, skilled)
    far_skilled, frr = rates_at_threshold(genuine, skilled, threshold)
    far_random, _ = rates_at_threshold(genuine, random, threshold)
    _, eer_random = eer_threshold(genuine, random)
    return {
        "eer_threshold": float(threshold),
        "eer_skilled": float(eer_skilled),
        "far_skilled": float(far_skilled),
        "frr": float(frr),
        "eer_random": float(eer_random),
        "far_random": float(far_random),
        "genuine_queries": int(len(genuine)),
        "skilled_forgeries": int(len(skilled)),
        "random_pairs": int(len(random)),
    }


results = {}
for name, path in MODELS.items():
    checkpoint = torch.load(path, map_location="cpu")
    model = SiameseSignatureModel(
        pretrained=False, embedding_dim=checkpoint["embedding_dim"]
    )
    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(DEVICE).eval()

    embeddings = embed_records(
        model, records,
        already_preprocessed=True, device=DEVICE, batch_size=BATCH_SIZE, workers=2,
    )
    scores = _evaluate_split(
        records, embeddings, [], REFERENCES, None, None,
        datasets=dataset_keys, dataset_of=dataset_of,
    )
    pooled = rate_block(
        scores["deep_genuine"], scores["deep_skilled"], scores["deep_random"]
    )

    # Per-dataset slices reuse the embeddings above, so this adds no forward passes.
    _, per_dataset, signers, counts = _collect(
        records, embeddings, [], dataset_keys, REFERENCES, None, None, None
    )
    per_ds = {}
    for key, value in per_dataset.items():
        entry = rate_block(
            value["deep_genuine"], value["deep_skilled"], value["deep_random"]
        )
        entry["signers_in_test"] = len(signers[key])
        entry["records_in_test"] = counts[key]
        per_ds[key] = entry

    results[name] = {
        "checkpoint": path,
        "validation_eer_skilled_from_checkpoint": checkpoint.get("selection_score"),
        "pooled_test": pooled,
        "per_dataset": per_ds,
    }
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    print(f"{name}: done")

out_dir = Path("/kaggle/working/eval_reports")
out_dir.mkdir(exist_ok=True)
report_path = out_dir / "best_eer_report.json"
report_path.write_text(json.dumps(results, indent=2))
print("wrote", report_path)
print(json.dumps(results, indent=2))

## 3. Compare base vs augmented

In [ ]:
# Cell 6 - side-by-side comparison. All rates are percentages; lower is better.
import pandas as pd

pooled_rows = []
for name, res in results.items():
    p = res["pooled_test"]
    pooled_rows.append({
        "model": name,
        "eer_skilled_%": 100 * p["eer_skilled"],
        "far_skilled_%": 100 * p["far_skilled"],
        "frr_%": 100 * p["frr"],
        "eer_random_%": 100 * p["eer_random"],
        "far_random_%": 100 * p["far_random"],
        "eer_threshold": p["eer_threshold"],
    })
pooled = pd.DataFrame(pooled_rows).set_index("model").round(2)
print("Pooled test - FAR / FRR at the best-EER operating point")
display(pooled)

ds_rows = []
for name, res in results.items():
    for dataset, m in res["per_dataset"].items():
        ds_rows.append({
            "model": name,
            "dataset": dataset,
            "signers": m["signers_in_test"],
            "eer_skilled_%": 100 * m["eer_skilled"],
            "far_skilled_%": 100 * m["far_skilled"],
            "frr_%": 100 * m["frr"],
        })
per_ds = pd.DataFrame(ds_rows).sort_values(["dataset", "model"]).reset_index(drop=True)
print("Per-dataset - FAR / FRR at the best-EER operating point")
display(per_ds.round(2))

## 4. Optional: the repo's own reporter

`training.evaluate` derives its decision threshold from the **validation** EER and applies
it unchanged to the test split, so its `far_*_at_threshold` / `frr_at_threshold` columns are
measured at the validation operating point, not at the test split's own EER. Run this only
if you want that validation-anchored reference block next to the numbers above.

In [ ]:
# Cell 7 (optional) - validation-anchored reference via training.evaluate (no target-FAR).
import json
from pathlib import Path

for name, path in MODELS.items():
    print(f"===== {name} =====")
    report = Path(f"/kaggle/working/eval_reports/{name}_evaluate.json")
    !python -m training.evaluate \
      --checkpoint {path} \
      --manifest {MANIFEST} \
      --already-preprocessed \
      --references {REFERENCES} \
      --device {DEVICE} \
      --batch-size {BATCH_SIZE} \
      --fusion-report {report}
    if report.is_file():
        print(json.dumps(json.loads(report.read_text())["test"], indent=2))

## Notes

* Outputs land in `/kaggle/working/eval_reports/`: `best_eer_report.json` (Cell 5) and one
  `<model>_evaluate.json` per model (Cell 7).
* `data/` and `checkpoints/` are gitignored, so neither the PNGs nor the weights are ever
  committed; both must be attached as Kaggle datasets (or produced by a training run).
* A cosine similarity is a score, not a calibrated probability. Read FAR / FRR as the
  operating-point behaviour of this protocol on this test split, not as a fraud rate.